In [1]:
import sys
sys.path.insert(0, '../')

import pandas as pd

from automed import *

In [2]:
titanic = pd.read_csv('../perf_logger/tests_data/titanic.csv', delimiter=';')

dataset = automed.Dataset(titanic.iloc[:200].copy())
dataset.set_label('label')

autom = automed.AutoMed(dataset=dataset)

In [3]:
print(autom.debug_load())
print(autom.json_pipeline())

None
{'step': 'MetaOrderedStep', 'name': 'MetaStep', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'RandomSplit', 'name': 'Split date to train and test set', 'description': 'Step description...', 'configuration': {'ratio': {'description': 'Split ratio', 'default': 0.2, 'value': 0.2}, 'random_state': {'description': 'Random state', 'default': 42, 'value': 42}}, 'children': [{'step': 'MetaStep', 'name': 'MetaStep', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'ActDropTextualColumn', 'name': 'Drop textual column', 'description': 'Step description...', 'configuration': {'ratio': {'description': "Description of the parameter's role", 'default': 0.8, 'value': 0.8}, 'random_state': {'description': "Description of the parameter's role", 'default': 12, 'value': 12}}, 'children': [{'step': 'ActSplitDate', 'name': 'Transform string column to date', 'description': 'Step description...', 'configuration': {}, 'children': [{'step'

In [4]:
pipeline = {
    'step': 'MetaOrderedStep',
    'children': [{
        'step': 'RandomSplit',
        'configuration': {
            'ratio': {
                'value': 0.3
            }
        }},
        {
            'step': 'MetaStep',
            'tag': 'cleaning'
        },
        {
            'step': 'MetaExplorerStep',
            'tag': 'learning'
        }]
}

autom.load_pipeline(pipeline)
print(autom.json_pipeline())


{'step': 'MetaOrderedStep', 'name': 'MetaStep', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'RandomSplit', 'name': 'Split date to train and test set', 'description': 'Step description...', 'configuration': {'ratio': {'description': 'Split ratio', 'default': 0.2, 'value': 0.3}, 'random_state': {'description': 'Random state', 'default': 42, 'value': 42}}, 'children': [{'step': 'MetaStep', 'name': 'MetaStep', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'ActDropTextualColumn', 'name': 'Drop textual column', 'description': 'Step description...', 'configuration': {'ratio': {'description': "Description of the parameter's role", 'default': 0.8, 'value': 0.8}, 'random_state': {'description': "Description of the parameter's role", 'default': 12, 'value': 12}}, 'children': [{'step': 'ActSplitDate', 'name': 'Transform string column to date', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'Ac

In [5]:
results = autom.run()
[ (r.model.sklearn_model, r.compute()) for r in results if r.model.sklearn_model is not None ]

Output()

[17:11:09] running step: MetaOrderedStep (steps=MetaStep,RandomSplit,MetaExplorerStep)

           running step: MetaStep                                                                                  
           (steps=ActTfIdf,ActDropDateColumn,ActOnehot,ActMeanColumn,ActDropTextualColumn,ActSplitDate,ActDropNumer
           icalColumn)

           running step: MetaExplorerStep                                                                          
           (steps=ActLogisticRegression,ActGaussianNb,ActRandomForest,ActSVM,ActXGBoost,ActKNN)

AttributeError: 'KNeighborsClassifier' object has no attribute 'sklearn_model'

In [ ]:
print(results[0].model.sklearn_model)
print(results[0].model.stack)

pm = results[0].model.pickle()
[ len(r.model.pickle()) for r in results ]

LogisticRegression(max_iter=1000, n_jobs=-1, random_state=42)
[(<function transform at 0x000001C1D2E65300>, ([('ind', 98.61428571428571), ('Pclass', 2.3857142857142857), ('Age', 27.67699115044248), ('SibSp', 0.6928571428571428), ('Parch', 0.4357142857142857), ('Fare', 30.458125000000006)],), {}), (<function transform at 0x000001C1D2E662A0>, (OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['Sex', 'Embarked']), {}), (<function transform at 0x000001C1D2E66700>, ([],), {}), (<function transform at 0x000001C1D2E65260>, ([('Name', TfidfVectorizer()), ('Ticket', TfidfVectorizer()), ('Cabin', TfidfVectorizer())],), {}), (<function transform at 0x000001C1D2E66340>, ([],), {}), (<function transform at 0x000001C1D2E65C60>, ([],), {}), (<function transform at 0x000001C1D2E65760>, ([],), {}), (<function sklearn_predict at 0x000001C1BA689260>, (LogisticRegression(max_iter=1000, n_jobs=-1, random_state=42),), {})]


[30168, 130721, 555334, 43205, 511616, 641735]

In [ ]:
import pickle

o = 200 # offset
n = 68  # # of samples
labels = titanic.iloc[o:(o+n)]['label']
predict_df = titanic.iloc[o:(o+n)].drop('label', axis=1).copy()

# labels = labels.reset_index()
predict_df.reset_index(inplace=True, drop=True)

m = pickle.loads(pm)
sum([ r == labels[o+i] for i, r in enumerate(m.run(predict_df)) ]) / n

0.7794117647058824

In [ ]:
final_boss_dataset = Dataset(titanic.copy())
final_boss_dataset.set_label('label')

final_boss_automed = AutoMed(final_boss_dataset)
final_boss_automed.debug_load()
final_boss_results = final_boss_automed.run()

[16:33:01] running step: MetaOrderedStep (steps=RandomSplit,MetaExplorerStep,MetaStep)

           running step: MetaStep                                                                                  
           (steps=ActDropNumericalColumn,ActMeanColumn,ActSplitDate,ActDropTextualColumn,ActTfIdf,ActDropDateColumn
           ,ActOnehot)

           running step: MetaStep (steps=)

           running step: MetaStep (steps=ActMinMaxScaler,ActRandomOverSampling)

[16:33:02] running step: MetaExplorerStep (steps=WrapGeneticGridSearch)

...

[16:35:01] created new generation: ActXGBoost (generation=3)

           running step: MetaExplorerStep (steps=ActXGBoost)

[16:35:30] created new generation: ActXGBoost (generation=4)

           running step: MetaExplorerStep (steps=ActXGBoost)

[16:36:14] finished all generations: ActXGBoost

In [ ]:
sum([ len(r.model.pickle()) for r in final_boss_results ])

105127560